# Seaborn for Statistical Data Visualization
**Summer of Science 2026 — CS03: Artificial Intelligence and Machine Learning**  
**Mohit Khyalia | IIT Bombay**

---

## Seaborn vs Matplotlib

Seaborn is built on top of Matplotlib. It provides higher-level functions for statistical plots that would take many lines to build in raw Matplotlib. The two key advantages for ML work:

1. **`hue` parameter** — split any plot by a categorical variable (e.g., by class label) in one line
2. **Built-in statistics** — plots like `boxplot` and `violinplot` compute and display distributions automatically

This notebook covers the Seaborn plots used in EDA across the mini-projects.

---

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

sns.set_theme(style='whitegrid', font_scale=1.05)
print('Seaborn version:', sns.__version__)

In [ ]:
# Build a synthetic loan dataset for this notebook
np.random.seed(42)
n = 300

education    = np.random.choice(['Undergraduate', 'Graduate', 'Postgraduate'], n, p=[0.25, 0.55, 0.20])
property_area= np.random.choice(['Urban', 'Semiurban', 'Rural'], n, p=[0.35, 0.40, 0.25])
employed     = np.random.choice(['Yes', 'No'], n, p=[0.75, 0.25])

income       = np.where(education == 'Postgraduate',
                   np.random.normal(95000, 20000, n),
                   np.where(education == 'Graduate',
                       np.random.normal(72000, 18000, n),
                       np.random.normal(48000, 15000, n)))

credit_score = np.random.normal(690, 45, n).clip(500, 850)
loan_amount  = np.random.exponential(150000, n).clip(50000, 600000)

# Approval probability depends on income, credit, education
approval_prob = 0.3 + 0.25*(income > 70000) + 0.2*(credit_score > 700) + 0.15*(education == 'Graduate')
loan_approved = np.random.binomial(1, approval_prob.clip(0, 1))

df = pd.DataFrame({
    'income':       income.round(0),
    'credit_score': credit_score.round(0),
    'loan_amount':  loan_amount.round(0),
    'education':    education,
    'property_area':property_area,
    'employed':     employed,
    'loan_approved':loan_approved
})

df['loan_status'] = df['loan_approved'].map({1: 'Approved', 0: 'Rejected'})

print(f'Dataset: {df.shape[0]} rows × {df.shape[1]} columns')
print(f'Approval rate: {df["loan_approved"].mean():.1%}')
print(df.head(3))

## 1. `countplot` — Categorical Distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, col in zip(axes, ['education', 'property_area', 'employed']):
    sns.countplot(x=col, hue='loan_status', data=df, ax=ax,
                  palette={'Approved': '#2CA02C', 'Rejected': '#C00000'},
                  alpha=0.85)
    ax.set_title(f'Loan Status by {col.replace("_", " ").title()}')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=15)
    ax.legend(title='Status', fontsize=8)

plt.suptitle('Categorical Feature Analysis', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('countplot_categorical.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Calculate approval rates per category — to quantify what the plot shows
for col in ['education', 'property_area', 'employed']:
    print(f'\nApproval rate by {col}:')
    rates = df.groupby(col)['loan_approved'].agg(['mean', 'count'])
    rates.columns = ['approval_rate', 'n']
    rates['approval_rate'] = rates['approval_rate'].map('{:.1%}'.format)
    print(rates.sort_values('approval_rate', ascending=False))

**Expected output (approximate):**
```
Approval rate by education:
               approval_rate    n
Graduate            61.2%    167
Postgraduate        58.3%     60
Undergraduate       43.5%     73

Approval rate by property_area:
             approval_rate    n
Semiurban        60.8%      121
Urban            56.2%      106
Rural            50.7%       73
```

**Observation:** Graduates have a higher approval rate than undergraduates, and Semiurban applicants have a marginally higher approval rate — consistent with the Loan Status Prediction project findings in the midterm report. These are trends to note, but with a sample this size they are not conclusive.

## 2. `histplot` — Numeric Feature Distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

numeric_cols = [
    ('income',       'Annual Income'),
    ('credit_score', 'Credit Score'),
    ('loan_amount',  'Loan Amount Requested')
]

for ax, (col, label) in zip(axes, numeric_cols):
    sns.histplot(data=df, x=col, hue='loan_status', ax=ax,
                 palette={'Approved': '#2CA02C', 'Rejected': '#C00000'},
                 bins=30, alpha=0.55, kde=True)
    ax.set_title(f'{label}')
    ax.set_xlabel('')
    ax.legend(title='', fontsize=8)

plt.suptitle('Numeric Feature Distributions by Loan Status', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('histplot_numeric.png', dpi=150, bbox_inches='tight')
plt.show()

**Observation:** The income distribution shows a clear rightward shift for approved applications — approved applicants tend to have higher incomes. Credit score shows a similar but less pronounced shift. Loan amount shows substantial overlap between classes, suggesting it is a weaker predictor on its own.

## 3. `heatmap` — Correlation Matrix

In [ ]:
numeric_df = df[['income', 'credit_score', 'loan_amount', 'loan_approved']]
corr_matrix = numeric_df.corr().round(2)

fig, ax = plt.subplots(figsize=(6, 5))

mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)   # hide upper triangle (redundant)

sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    vmin=-1, vmax=1,
    square=True,
    linewidths=0.5,
    ax=ax
)

ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150)
plt.show()

# Print correlations with the target
print('\nCorrelations with loan_approved:')
print(corr_matrix['loan_approved'].sort_values(ascending=False))

**Expected output (approximate):**
```
Correlations with loan_approved:
loan_approved    1.00
credit_score     0.28
income           0.26
loan_amount      0.03
Name: loan_approved, dtype: float64
```

**Observation:** Credit score and income have a moderate positive correlation with approval (0.28 and 0.26). Loan amount is nearly uncorrelated with approval (0.03), confirming the histplot finding. Note the correlation is between income and credit score as well — multicollinearity like this can affect the stability of linear model coefficients.

## 4. `boxplot` — Comparing Distributions Across Groups

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

sns.boxplot(x='education', y='income', hue='loan_status', data=df,
            palette={'Approved': '#2CA02C', 'Rejected': '#C00000'},
            ax=axes[0], width=0.5)
axes[0].set_title('Income Distribution by Education and Loan Status')
axes[0].set_xlabel('Education')
axes[0].set_ylabel('Annual Income')
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))
axes[0].legend(title='Status', fontsize=8)

sns.boxplot(x='property_area', y='credit_score', hue='loan_status', data=df,
            palette={'Approved': '#2CA02C', 'Rejected': '#C00000'},
            ax=axes[1], width=0.5)
axes[1].set_title('Credit Score by Property Area and Loan Status')
axes[1].set_xlabel('Property Area')
axes[1].set_ylabel('Credit Score')
axes[1].legend(title='Status', fontsize=8)

plt.tight_layout()
plt.savefig('boxplot_groups.png', dpi=150)
plt.show()

**Observation:** Boxplots make the income gap between approved and rejected applicants visible across all education levels. The median approved income is consistently higher than the median rejected income within each education group — suggesting income has predictive power independent of education level.

## 5. `pairplot` — All Pairwise Relationships at Once

In [ ]:
# pairplot is most useful for datasets with a small number of features
# for large feature sets it becomes unreadable

sample_df = df[['income', 'credit_score', 'loan_amount', 'loan_status']].sample(150, random_state=0)

pair_grid = sns.pairplot(
    sample_df,
    hue='loan_status',
    palette={'Approved': '#2CA02C', 'Rejected': '#C00000'},
    diag_kind='kde',       # diagonal: kernel density estimate
    plot_kws={'alpha': 0.4, 's': 20},
    diag_kws={'alpha': 0.6}
)
pair_grid.figure.suptitle('Pairplot: Numeric Features Coloured by Loan Status',
                           y=1.02, fontsize=11)
plt.savefig('pairplot.png', dpi=130, bbox_inches='tight')
plt.show()

print('Note: pairplot with many features becomes cluttered quickly.')
print('For datasets with >6-8 features, prefer individual scatter plots')
print('or a correlation heatmap to get the same information more compactly.')

## 6. Seaborn + Matplotlib Together — Full EDA Summary Figure

In [ ]:
fig = plt.figure(figsize=(14, 9))
fig.suptitle('Complete EDA Summary — Loan Dataset', fontsize=13, y=1.01)

# Row 1: Categorical features
ax1 = fig.add_subplot(2, 3, 1)
sns.countplot(x='education', hue='loan_status', data=df, ax=ax1,
              palette={'Approved': '#2CA02C', 'Rejected': '#C00000'}, alpha=0.85)
ax1.set_title('By Education')
ax1.set_xlabel('')
ax1.tick_params(axis='x', rotation=20)
ax1.get_legend().remove()

ax2 = fig.add_subplot(2, 3, 2)
sns.countplot(x='property_area', hue='loan_status', data=df, ax=ax2,
              palette={'Approved': '#2CA02C', 'Rejected': '#C00000'}, alpha=0.85)
ax2.set_title('By Property Area')
ax2.set_xlabel('')
ax2.get_legend().remove()

ax3 = fig.add_subplot(2, 3, 3)
approved_vc = df['loan_status'].value_counts()
colors = ['#2CA02C', '#C00000']
ax3.pie(approved_vc.values, labels=approved_vc.index, colors=colors,
        autopct='%1.1f%%', startangle=90)
ax3.set_title('Class Balance')

# Row 2: Numeric features
ax4 = fig.add_subplot(2, 3, 4)
sns.histplot(data=df, x='income', hue='loan_status', ax=ax4, bins=25,
             palette={'Approved': '#2CA02C', 'Rejected': '#C00000'}, alpha=0.55, kde=True)
ax4.set_title('Income Distribution')
ax4.set_xlabel('Income')
ax4.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))
ax4.get_legend().remove()

ax5 = fig.add_subplot(2, 3, 5)
sns.histplot(data=df, x='credit_score', hue='loan_status', ax=ax5, bins=25,
             palette={'Approved': '#2CA02C', 'Rejected': '#C00000'}, alpha=0.55, kde=True)
ax5.set_title('Credit Score Distribution')
ax5.set_xlabel('Credit Score')
ax5.get_legend().remove()

ax6 = fig.add_subplot(2, 3, 6)
corr = df[['income', 'credit_score', 'loan_amount', 'loan_approved']].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, ax=ax6, cbar=False,
            annot_kws={'size': 9})
ax6.set_title('Correlation Matrix')

# Shared legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#2CA02C', label='Approved'),
                   Patch(facecolor='#C00000', label='Rejected')]
fig.legend(handles=legend_elements, loc='lower center', ncol=2,
           fontsize=10, bbox_to_anchor=(0.5, -0.02))

plt.tight_layout()
plt.savefig('eda_summary_seaborn.png', dpi=150, bbox_inches='tight')
plt.show()

---

## Summary — Seaborn vs Matplotlib Cheatsheet

| Task | Seaborn | Matplotlib equivalent |
|---|---|---|
| Count categories | `sns.countplot(x, hue)` | Multiple `ax.bar()` calls |
| Distributions | `sns.histplot(kde=True)` | `ax.hist()` + manual KDE |
| Correlation | `sns.heatmap(df.corr())` | `ax.imshow()` + manual annotations |
| Group comparison | `sns.boxplot(x, y, hue)` | Multiple `ax.boxplot()` calls |
| All pairs | `sns.pairplot(hue)` | Grid of `ax.scatter()` calls |

**Key takeaway:** Use Seaborn when `hue` (splitting by class) matters and the plot type is statistical. Use Matplotlib directly when fine-grained control over layout, annotations, or non-standard plot types is needed. Both are typically used together in the same figure.

---
*Notebook — Mohit Khyalia, Summer of Science 2026, IIT Bombay*